<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Machine Learning with Spark MLlib: Regression (Databricks)

*Session 5 · Notebook 04.03 · Lecture · Coach version · Databricks*

## Overview

This notebook builds a full machine-learning model **at scale** with **Spark MLlib**, the machine-learning library that runs on Spark DataFrames. It is the same workflow you already know from scikit-learn (explore, prepare features, scale, split, fit, predict, evaluate, tune), expressed in Spark so it runs on data too big for one machine.

We predict **median house value** on the California housing data. Along the way we explore the data with plots, engineer features, assemble them into a feature vector, standardise, split, fit a **linear regression with elastic-net regularisation**, evaluate with RMSE, MAE and R-squared, visualise the results.

## Learning Objectives

By the end of this notebook you will be able to:

- Explore a Spark DataFrame with summary statistics and plots.
- Explain and perform feature engineering, and say why MLlib needs the features in a single vector column.
- Scale features and split the data in Spark.
- Explain elastic-net regularisation and fit a `LinearRegression` with it.
- Evaluate a regression (RMSE, MAE, R-squared) and visualise coefficients and predictions.


## Prerequisites

- Notebook 04.02 (PySpark fundamentals: SparkSession, DataFrames).
- The regression notebooks (02.02 to 02.07): the concepts (features, scaling, train/test split, RMSE/R-squared, regularisation) are identical here.
- A **Spark environment** (see the setup note).

## Running this notebook on Databricks

This is the **Databricks** version. On Databricks you do not install Spark or create a session yourself:

- A **SparkSession is already provided** as `spark` (the `getOrCreate()` calls below simply return it).
- **No `pip install pyspark`** is needed; the cluster provides Spark.
- **Upload the data once to DBFS**, for example to `/FileStore/cbs_datasets/Session_5/`, via *Catalog / DBFS / Upload* (or a Unity Catalog **Volume**), then point `DATA_PATH` below at it. Spark reads it with the `dbfs:/` scheme.
- Do **not** call `spark.stop()`; the cluster manages the session (the stop cells below are commented out).
- Tip: you can use Databricks' built-in `display(df)` instead of `df.show()` for richer tables and charts.

In [ ]:
DATA_PATH = "dbfs:/FileStore/cbs_datasets/Session_5/"   # DBFS location where you uploaded the data

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is Spark MLlib?](#sec2)
3. [Load the California housing data](#sec3)
4. [Explore the data](#sec4)
5. [Feature engineering](#sec5)
6. [Assemble features into a vector (and why)](#sec6)
7. [Standardise the features](#sec7)
8. [Train / test split](#sec8)
9. [Elastic-net regularisation explained](#sec9)
10. [Train the model and read the coefficients](#sec10)
11. [Predict and evaluate (with plots)](#sec11)
12. [Key Takeaways](#takeaways)

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Everything in the regression and classification notebooks assumed the data fits in memory. In production, a risk model may be trained on **hundreds of millions of rows** (every transaction, every account-month). Spark MLlib lets you run the *same* modelling workflow on that scale, inside the same distributed platform that already holds the data, so you avoid sampling the data down just to fit it on one machine. The skills transfer directly: this is your scikit-learn knowledge, expressed for a cluster.

<a id="sec2"></a>
# Section 2: What is Spark MLlib?

**Definition:** MLlib is Spark's machine-learning library. It provides the familiar estimators (linear and logistic regression, decision trees, random forests, gradient-boosted trees, k-means, and more) that train on Spark DataFrames across a cluster.

**Example:** fitting a linear regression on a billion-row table: MLlib distributes the fitting across the cluster and returns one model, using the same `fit` / `transform` idea as scikit-learn.

**Analogy:** it is the scikit-learn toolbox, re-built to work in Spark's distributed kitchen instead of on a single benchtop.

**Explanation (the one big difference from scikit-learn):**

- In scikit-learn, `X` is a table of feature columns. In **MLlib, every model expects all the features packed into a single vector column**, which you build with a `VectorAssembler` (Section 6 explains why).
- MLlib estimators take `featuresCol` and `labelCol` arguments naming those columns, and, like Spark generally, transformations are **lazy** until an action runs.
- The overall recipe is: **explore, engineer features, assemble into a vector, scale, split, fit, predict, evaluate, tune.**

<a id="sec3"></a>
# Section 3: Load the California housing data

The file `cal_housing.data` has **no header**, so we define an explicit schema (column names and types). The columns are geographic and demographic features of California districts, and the target `medhv` is the median house value.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

rnd_seed = 42
# On Databricks the cluster already provides the `spark` session (no import or creation needed).
# Local / non-Databricks environments would instead need:
#   from pyspark.sql import SparkSession
#   spark = SparkSession.builder.appName("cal_housing_mllib").getOrCreate()

In [ ]:
# The file has no header row, so we name the columns (and keep a Spark schema for the Spark scenarios).
schema = StructType([
    StructField("long", FloatType(), True),      # longitude of the district
    StructField("lat", FloatType(), True),       # latitude of the district
    StructField("medage", FloatType(), True),    # median house age
    StructField("totrooms", FloatType(), True),  # total rooms in the district
    StructField("totbdrms", FloatType(), True),  # total bedrooms in the district
    StructField("pop", FloatType(), True),       # population
    StructField("houshlds", FloatType(), True),  # number of households
    StructField("medinc", FloatType(), True),    # median income
    StructField("medhv", FloatType(), True),     # median house value  <-- the target
])
URL = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/"
# Scenario A (active): pandas reads the headerless file over HTTPS, then hands it to Spark.
cols = [f.name for f in schema.fields]
pdf = pd.read_csv(URL + "cal_housing.data", header=None, names=cols)
cal_housing_df = spark.createDataFrame(pdf)

# Other ways to load the same data (uncomment the scenario you need):
# Scenario B - Spark reads directly from S3 with the schema (s3:// works on Databricks):
# cal_housing_df = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/cal_housing.data", schema=schema)
# Scenario C - pandas over HTTPS via a config file, then to Spark:
# from config import session_datasets_http
# cal_housing_df = spark.createDataFrame(pd.read_csv(session_datasets_http["cal_housing"], header=None, names=cols))
# Scenario D - Spark reads s3:// via a config file:
# from config import session_datasets
# cal_housing_df = spark.read.csv(session_datasets["cal_housing"], schema=schema)
# Scenario E - a file you uploaded to DBFS:
# cal_housing_df = spark.read.csv("dbfs:/FileStore/cbs_datasets/Session_5/cal_housing.data", schema=schema)

cal_housing_df.show(3)

<a id="sec4"></a>
# Section 4: Explore the data

Before modelling, understand the data: its size, its types, its summary statistics, and the shape of the target and its relationships. In Spark we compute the summaries **on the cluster**, then pull the (small) results into pandas to plot them. Never `toPandas()` a huge DataFrame in full; summarise or sample first.

In [ ]:
# Size and schema
print("Rows:", cal_housing_df.count(), "| Columns:", len(cal_housing_df.columns))
cal_housing_df.printSchema()

In [ ]:
# Summary statistics (computed in Spark, shown as a small pandas table)
cal_housing_df.describe().toPandas().set_index("summary").T

In [ ]:
# Distribution of the target (median house value).
# .toPandas() is safe here because we first select a single column.
medhv_pd = cal_housing_df.select("medhv").toPandas()

plt.figure(figsize=(8, 4))
plt.hist(medhv_pd["medhv"], bins=40, color="steelblue")
plt.xlabel("median house value"); plt.ylabel("count")
plt.title("Target distribution: note the spike at the top (values are capped)")
plt.show()

In [ ]:
# Relationship between median income and median house value (a strong predictor).
# We sample to keep the scatter light, then plot with pandas.
sample_pd = cal_housing_df.select("medinc", "medhv").sample(fraction=0.1, seed=rnd_seed).toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(sample_pd["medinc"], sample_pd["medhv"], s=8, alpha=0.3)
plt.xlabel("median income"); plt.ylabel("median house value")
plt.title("Higher income districts tend to have higher house values")
plt.show()

In [ ]:
# How many districts share each median-house-age value? A quick grouped count.
cal_housing_df.groupBy("medage").count().sort("count", ascending=False).show(5)

What the exploration tells us: the target is right-skewed and **capped** at the top (a flat spike), median income is clearly related to house value (a useful feature), and the raw totals (`totrooms`, `totbdrms`, `pop`) are district-wide sums that will be more meaningful once turned into per-household ratios, which is the next step.

<a id="sec5"></a>
# Section 5: Feature engineering

**What is feature engineering?** It is creating better input columns from the raw ones so the model has more directly useful signal. Raw district totals like `totrooms` depend on how big the district is; dividing by the number of households turns them into meaningful **per-household** quantities that compare fairly across districts.

**What we will do here:**

1. **Rescale the target** `medhv` into units of 100,000, so the numbers (and later the error metrics) are easier to read.
2. **Create ratio features** that are more informative than the raw totals:
   - `rooms_per_household` = total rooms / households
   - `pop_per_household` = population / households
   - `bedrooms_per_room` = total bedrooms / total rooms
3. **Keep only** the target and the columns we will model on.

In [ ]:
# 1. Rescale the target into units of 100,000 (so 2.5 means $250,000)
cal_df = cal_housing_df.withColumn("medhv", col("medhv") / 100000)

# 2. Build ratio features. F.round keeps them tidy; each withColumn adds one new column.
cal_df = (cal_df
          .withColumn("rooms_per_household", F.round(col("totrooms") / col("houshlds"), 2))  # rooms per household
          .withColumn("pop_per_household",  F.round(col("pop")      / col("houshlds"), 2))  # people per household
          .withColumn("bedrooms_per_room",  F.round(col("totbdrms") / col("totrooms"), 2))) # bedroom share

# 3. Keep the target plus the features we will actually use
cal_df = cal_df.select("medhv", "pop", "houshlds", "medinc",
                       "rooms_per_household", "pop_per_household", "bedrooms_per_room")
cal_df.show(5)

<a id="sec6"></a>
# Section 6: Assemble features into a vector (and why)

**Why a single vector column?** scikit-learn accepts a table of feature columns directly. Spark MLlib does not: for speed and consistency across a cluster, **every MLlib model expects all the features packed into one column, where each row holds a vector of that row's feature values**. The `VectorAssembler` does this packing: it takes a list of columns and produces one `features` column of vectors. This is the one extra step in every MLlib pipeline compared with scikit-learn.

In [ ]:
# List the feature columns we want to combine (we leave out lat/long here)
featurecols = ["pop", "houshlds", "medinc",
               "rooms_per_household", "pop_per_household", "bedrooms_per_room"]

# VectorAssembler packs those columns into a single 'features' vector column
assembler = VectorAssembler(inputCols=featurecols, outputCol="features")
assembled_df = assembler.transform(cal_df)      # transform ADDS the new 'features' column

# Each row's 'features' is now a vector of its 6 feature values
assembled_df.select("features", "medhv").show(5, truncate=False)

> **Note - encoding categorical features.** This dataset is all numeric, so no encoding is needed here. When you *do* have a categorical (string) column, MLlib models cannot read the text directly, so you convert it to numbers **before** assembling, in two steps:
>
> 1. **`StringIndexer`** (PySpark's equivalent of a label encoder) maps each distinct category to a numeric index, for example `"NEAR BAY" -> 0.0`, `"INLAND" -> 1.0`, `"ISLAND" -> 2.0`. `handleInvalid="keep"` sends any category it did not see during fit into an extra bucket instead of raising an error at prediction time.
> 2. **`OneHotEncoder`** turns that single index column into a sparse 0/1 vector with one position per category. This stops the model from reading the indices as an order or magnitude (otherwise it would think `"ISLAND"` (2.0) is somehow "twice" `"INLAND"` (1.0), which is meaningless for categories).
>
> Then **add the encoder's output column to the `VectorAssembler`'s `inputCols`** so the one-hot vector becomes part of the feature vector the model trains on. Put all of these stages **in the Pipeline before the assembler and scaler** so they are fit on the training split only (the same no-leakage rule as the scaler).
>
> ```python
> from pyspark.ml.feature import StringIndexer, OneHotEncoder
> indexer = StringIndexer(inputCol="ocean_proximity", outputCol="ocean_idx", handleInvalid="keep")
> encoder = OneHotEncoder(inputCols=["ocean_idx"], outputCols=["ocean_ohe"])
> assembler = VectorAssembler(inputCols=[*numeric_cols, "ocean_ohe"], outputCol="features")
> pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])
> ```
>
> Docs to review later: [StringIndexer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html) (label encoding) and [OneHotEncoder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.OneHotEncoder.html).

<a id="sec7"></a>
# Section 7: Standardise the features

As with distance- and penalty-based models in scikit-learn, we standardise so no feature dominates just because of its units. MLlib's `StandardScaler` is an estimator: `fit` learns each feature's mean and standard deviation, then `transform` applies them. To avoid **data leakage** we only *create* the scaler here and defer fitting it until **after** the train/test split, so it learns from the training data alone (Section 8).

In [ ]:
# Create the scaler (an estimator). IMPORTANT: to avoid data leakage its mean/std must be
# LEARNED from the TRAINING data only, so we fit it AFTER the train/test split (Section 8), not on the full data.
standard_scaler = StandardScaler(inputCol="features", outputCol="features_scaled")

<a id="sec8"></a>
# Section 8: Train / test split

`randomSplit` is Spark's equivalent of `train_test_split`. We pass the proportions and a seed for reproducibility.

In [ ]:
# Split FIRST, on the assembled (but not-yet-scaled) data
train_assembled, test_assembled = assembled_df.randomSplit([0.8, 0.2], seed=rnd_seed)

# Fit the scaler on the TRAINING data ONLY, then apply it to both sets.
# This prevents leakage: the test set's statistics never influence the scaling.
scaler_model = standard_scaler.fit(train_assembled)
train_data = scaler_model.transform(train_assembled)
test_data = scaler_model.transform(test_assembled)
print("Train rows:", train_data.count(), "| Test rows:", test_data.count())

<a id="sec9"></a>
# Section 9: Elastic-net regularisation explained

Recall from the Ridge and Lasso notebooks that **regularisation** adds a penalty on the size of the coefficients to curb overfitting. There are two classic penalties:

- **L2 (Ridge):** penalises the sum of *squared* coefficients. It shrinks all coefficients smoothly toward zero but never exactly to zero.
- **L1 (Lasso):** penalises the sum of *absolute* coefficients. It can drive some coefficients to exactly zero, performing feature selection.

**Elastic net** is a blend of the two, and Spark's `LinearRegression` exposes it through two settings:

- **`regParam`** is the overall **strength** of the penalty (like `alpha` in scikit-learn's Ridge/Lasso). Larger means more shrinkage.
- **`elasticNetParam`** is the **mix** between L1 and L2, from 0 to 1:
  - `0.0` = pure **L2** (Ridge),
  - `1.0` = pure **L1** (Lasso),
  - in between = a mixture (for example `0.5` is half and half).

Elastic net is useful when you have many, possibly correlated, features: the L2 part keeps correlated features stable, while the L1 part still removes the useless ones. We start with `regParam=0.3` and `elasticNetParam=0.8` (mostly Lasso).

<a id="sec10"></a>
# Section 10: Train the model and read the coefficients

In [ ]:
# Define the model: name the (scaled) features column and the label column,
# and set the elastic-net penalty (strength and L1/L2 mix)
lin_reg = LinearRegression(featuresCol="features_scaled",   # the standardised feature vector
                           labelCol="medhv",                # what we predict
                           predictionCol="predmedhv",       # where predictions will go
                           maxIter=10,                      # optimisation iterations
                           regParam=0.3,                    # penalty strength
                           elasticNetParam=0.8)             # 0=L2(Ridge), 1=L1(Lasso), 0.8=mostly Lasso

# fit() runs the (distributed) training and returns a fitted model
linear_model = lin_reg.fit(train_data)
print("Model fitted.")

In [ ]:
# Collect the intercept and one coefficient per feature into a small pandas table
coeff_df = pd.DataFrame({
    "feature": featurecols,
    "coefficient": linear_model.coefficients.toArray(),   # .toArray() turns the Spark vector into numpy
})
print("Intercept:", round(linear_model.intercept, 3))
print(coeff_df.round(3))

In [ ]:
# Plot the coefficients so their relative influence is easy to see
order = coeff_df.reindex(coeff_df["coefficient"].abs().sort_values().index)
colors = ["tab:green" if v > 0 else "tab:red" for v in order["coefficient"]]

plt.figure(figsize=(8, 4))
plt.barh(order["feature"], order["coefficient"], color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.title("Linear regression coefficients (on standardised features)")
plt.xlabel("effect on median house value (green = raises, red = lowers)")
plt.tight_layout()
plt.show()
# With elastic net, weak features may be shrunk to (near) zero; median income is usually the strongest.

<a id="sec11"></a>
# Section 11: Predict and evaluate (with plots)

`transform` generates predictions on the test set (adding the `predmedhv` column). We evaluate with `RegressionEvaluator`, then plot predicted against actual values, the clearest visual check of a regression.

In [ ]:
# transform() = predict on the test set (adds the 'predmedhv' column)
predictions = linear_model.transform(test_data)
predictions.select("medinc", "predmedhv", "medhv").show(8)

In [ ]:
# Evaluate with RMSE, MAE and R2 (one evaluator per metric)
pred_and_labels = predictions.select("predmedhv", "medhv")
for metric in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(predictionCol="predmedhv", labelCol="medhv", metricName=metric)
    print(f"{metric.upper()}: {evaluator.evaluate(pred_and_labels):.4f}")

In [ ]:
# Predicted vs actual: points near the diagonal are good predictions
pa = pred_and_labels.sample(fraction=0.2, seed=rnd_seed).toPandas()

plt.figure(figsize=(6, 6))
plt.scatter(pa["medhv"], pa["predmedhv"], s=8, alpha=0.3)
lims = [pa[["medhv", "predmedhv"]].min().min(), pa[["medhv", "predmedhv"]].max().max()]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("actual median house value"); plt.ylabel("predicted")
plt.title("Predicted vs actual")
plt.legend()
plt.show()

In [ ]:
# End the SparkSession to release the cluster resources
# On Databricks, do NOT stop the shared SparkSession (the cluster manages it):
# spark.stop()

<a id="takeaways"></a>
## Key Takeaways

| scikit-learn | Spark MLlib equivalent |
|---|---|
| `X` as many columns | one `features` vector column via `VectorAssembler` |
| `StandardScaler().fit_transform(X)` | `StandardScaler(...).fit(df).transform(df)` |
| `train_test_split` | `df.randomSplit([0.8, 0.2], seed=...)` |
| `LinearRegression().fit(X, y)` | `LinearRegression(featuresCol=..., labelCol=...).fit(df)` |
| `model.predict(X_test)` | `model.transform(test_df)` (adds a prediction column) |
| `r2_score`, RMSE | `RegressionEvaluator(metricName="r2" / "rmse")` |
| `Ridge` / `Lasso` | `regParam` + `elasticNetParam` on the estimator |

**The one-line lesson:** Spark MLlib is scikit-learn's workflow for the cluster. The habits to remember are that **every model consumes a single assembled feature vector column**, and that it otherwise follows the same assemble, fit, transform, evaluate workflow you already know.

MLlib also covers classification (see notebook 04.05) and clustering, all following this same assemble-fit-transform-evaluate pattern.

## Conclusion

You can now run an end-to-end regression at scale with Spark MLlib: explore with plots, engineer features, assemble and scale them, split, fit an elastic-net linear regression, evaluate and visualise it. Notebook 04.05 applies the same pattern to classification with decision trees and random forests.

<a id="reading"></a>
## Further Reading & Resources

- [Spark MLlib guide](https://spark.apache.org/docs/latest/ml-guide.html)
- [`VectorAssembler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html), [`LinearRegression`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html), [`RegressionEvaluator`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)
